In [ ]:
#!pip install dotenv
#!pip install -qU langchain-teddynote


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from dotenv import load_dotenv
#import os
from langchain_teddynote import logging

In [12]:
load_dotenv()

logging.langsmith("CH02-Prompt")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH02-Prompt


In [14]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()

In [16]:
from langchain_core.prompts import PromptTemplate

# template 정의. {country}는 변수로, 이후 값이 들어갈 자리 의미. 
template = "{country}의 수도는 어디?"

In [22]:
# from_template 메서드를 이용하여 PromptTemplate 객체 생성.
prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디?')

In [ ]:
# # prompt 생성, format 메서드를 이용하여 변수에 값을 넣어줌. 
# prompt = prompt.format(country="대한민국")
# prompt

'대한민국의 수도는 어디?'

In [24]:
# chain 생성.
chain = prompt | llm

# country 변수에 입력된 값이 자동으로 치환되어 수행됨. 
chain.invoke("일본").content

'일본의 수도는 도쿄입니다.'

In [26]:
# template 정의
template = "{country}의 수도는 어디?"

# PromptTemplate 객체를 활용하여 prompt_template 생성. 
prompt = PromptTemplate(
    template=template,
    input_variables=["country"],
)

prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디?')

In [28]:
prompt.format(country="대한민국")

'대한민국의 수도는 어디?'

In [30]:
template = "{country1}과 {country2}의 수도는 각각 어디?"

prompt=PromptTemplate(
    template=template,
    input_variables=["country1"],
    partial_variables={
        "country2": "미국"
    }
)

prompt

PromptTemplate(input_variables=['country1'], input_types={}, partial_variables={'country2': '미국'}, template='{country1}과 {country2}의 수도는 각각 어디?')

In [32]:
prompt.format(country1="대한민국")

'대한민국과 미국의 수도는 각각 어디?'

In [33]:
prompt_partial = prompt.partial(country2="일본")
prompt_partial

PromptTemplate(input_variables=['country1'], input_types={}, partial_variables={'country2': '일본'}, template='{country1}과 {country2}의 수도는 각각 어디?')

In [34]:
prompt_partial.format(country1="대한민국")

'대한민국과 일본의 수도는 각각 어디?'

In [35]:
chain = prompt_partial | llm

chain.invoke("대한민국").content

'대한민국의 수도는 서울이며, 일본의 수도는 도쿄입니다.'

In [36]:
chain.invoke({"country1": "일본", "country2": "중국"}).content

'일본의 수도는 도쿄이고 중국의 수도는 베이징입니다.'

In [37]:
from datetime import datetime

# 오늘 날짜 출력.
datetime.now().strftime("%B %d")

'September 15'

In [38]:
def get_today():
    return datetime.now().strftime("%B %d")

In [39]:
prompt = PromptTemplate(
    template="오늘의 날짜는 {today}입니다. 오늘이 생일인 애니 캐릭터 {n}명을 나열해 주세요. 생년월일을 표기해주세요. ",
    input_variables=["n"],
    partial_variables={
        "today": get_today  # dictionary 형태로 partial_variables를 전달
    }
)

In [40]:
prompt.format(n=3)

'오늘의 날짜는 September 15입니다. 오늘이 생일인 애니 캐릭터 3명을 나열해 주세요. 생년월일을 표기해주세요. '

In [41]:
chain = prompt | llm
print(chain.invoke(3).content)

1. 미소노 히비키 (미라이 다키 신세키) - 9월 15일
2. 니시조노카나에 (라브라이브! 선샤인!!) - 9월 15일
3. 이구시마 이나바 (바디 스윗) - 9월 15일


In [42]:
print(chain.invoke({"today": "December 27", "n": 3}).content)

1. 미사토 아야세 (Neon Genesis Evangelion) - December 27, 2001년
2. 에보이 (One Piece) - December 27, 2002년
3. 리나 이발슈타인 (바이시티) - December 27, 1997년


In [44]:
from langchain_core.prompts import load_prompt

prompt = load_prompt("prompts/fruit_color.yaml", encoding="utf-8")
prompt

PromptTemplate(input_variables=['fruit'], input_types={}, partial_variables={}, template='{fruit}의 색깔이 뭐야?')

In [45]:
prompt.format(fruit="사과")

'사과의 색깔이 뭐야?'

In [47]:
prompt2 = load_prompt("prompts/capital.yaml", encoding="utf-8")
print(prompt2.format(country="일본"))

일본의 수도에 대해서 알려주세요.
수도의 특징을 다음의 양식에 맞게 정리해 주세요.
300자 내외로 작성해 주세요.
한글로 작성해 주세요.
----
[양식]
1. 면적
2. 인구
3. 역사적 장소
4. 특산품

#Answer:



In [49]:
from langchain_core.output_parsers import StrOutputParser
from langchain_teddynote.messages import stream_response

chain = prompt2 | ChatOpenAI(model="gpt-5.6-luna", temperature=0.1) | StrOutputParser()

answer = chain.stream({"country": "일본"})
stream_response(answer)

#Answer:

1. 면적: 일본의 수도 도쿄도는 약 2,194㎢이며, 23개 특별구와 다마 지역, 이즈·오가사와라 제도로 이루어져 있습니다.  
2. 인구: 약 1,400만 명이 거주하며, 수도권까지 포함하면 3,700만 명이 넘는 세계적인 대도시입니다.  
3. 역사적 장소: 에도 시대의 흔적이 남은 황궁, 아사쿠사의 센소지, 메이지 신궁, 우에노 공원 등이 유명합니다.  
4. 특산품: 에도마에 스시, 몬자야키, 도쿄 바나나, 닌교야키 등이 대표적입니다. 전통과 현대 문화가 조화를 이루는 도시입니다.